# Load the data

In [1]:
import pandas as pd

df = pd.read_parquet('combined.parquet')
print(df.shape)
df.head()

(6940494, 31)


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN_CITY_NAME,DEST_CITY_NAME,CRS_DEP_TIME,...,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,ARR_DELAY,DIVERTED,AIR_TIME
0,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"New York, NY","Los Angeles, CA",659,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Los Angeles, CA","New York, NY",2200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Madison, WI","Charlotte, NC",644,...,20.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Philadelphia, PA","Las Vegas, NV",1855,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Chicago, IL","Phoenix, AZ",830,...,0.0,0.0,44.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN


In [2]:
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])
df = df.drop_duplicates()
print(df.shape)

C:\Users\denni\AppData\Local\Temp\ipykernel_15524\1529318865.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])


(6400744, 31)


# Clean rows

Remove cancelled flights and rows with a missing target.

In [3]:
df = df[df['CANCELLED'] == 0].copy()
print('After dropping cancellations:', df.shape)

After dropping cancellations: (6304794, 31)


In [5]:
df = df.dropna(subset=['DEP_DELAY'])
df['DEP_DELAY'] = df['DEP_DELAY'].astype(int)
print('After dropping missing target:', df.shape)
df['DEP_DELAY'].value_counts()

After dropping missing target: (6304794, 31)


DEP_DELAY
-5       427499
-6       396457
-4       392880
-3       360573
-7       345424
          ...  
 2515         1
 1865         1
 1951         1
 2126         1
 1481         1
Name: count, Length: 1871, dtype: int64

# Feature engineering

Split `"City, ST"` into separate city/state columns, and extract `DEP_HOUR` (0–23) from the HHMM `CRS_DEP_TIME`.

In [ ]:
df[['ORIGIN_CITY', 'ORIGIN_STATE']] = df['ORIGIN_CITY_NAME'].str.split(',', expand=True)
df[['DEST_CITY', 'DEST_STATE']] = df['DEST_CITY_NAME'].str.split(',', expand=True)

df['ORIGIN_STATE'] = df['ORIGIN_STATE'].str.strip()

# Drop columns we don't need

Drop columns only known after the flight, redundant/constant columns, and high-cardinality city columns (keeping state instead).

In [ ]:
drop_cols = [
    # only known after the flight lands
    'DEP_TIME','WHEELS_OFF', 'WHEELS_ON',
    'ARR_TIME', 'ARR_DEL15', 'AIR_TIME',
    'CARRIER_DELAY',

    # already filtered or not useful
    'CANCELLED', 'CANCELLATION_CODE',
    , 'DEST_CITY_MARKET_ID',

    # redundant
    'FL_DATE',
    'CRS_DEP_TIME', 'CRS_ARR_TIME',

    # replaced by state. Too many columns for one-hot
    'ORIGIN_CITY_NAME', 'DEST_CITY_NAME'
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df.shape)
df.head()

(6287183, 8)


,MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,ARR_DEL15,DISTANCE,ORIGIN_STATE,DEST_STATE,DEP_HOUR
0,1,3,AA,0,2475.0,NY,CA,6
1,1,3,AA,0,2475.0,CA,NY,22
2,1,3,AA,1,708.0,WI,NC,6
3,1,3,AA,0,2176.0,PA,NV,18
4,1,3,AA,1,1440.0,IL,AZ,8


# One-hot encode categoricals

Convert carrier and state columns into 0/1 columns so the model can use them.

In [8]:
df_model = pd.get_dummies(
    df,
    columns=['OP_UNIQUE_CARRIER', 'ORIGIN_STATE', 'DEST_STATE'],
    drop_first=True,
    dtype=int,
)
print(df_model.shape)
df_model.head()

(6287183, 122)


,MONTH,DAY_OF_WEEK,ARR_DEL15,DISTANCE,DEP_HOUR,OP_UNIQUE_CARRIER_AS,OP_UNIQUE_CARRIER_B6,OP_UNIQUE_CARRIER_DL,OP_UNIQUE_CARRIER_F9,OP_UNIQUE_CARRIER_G4,...,DEST_STATE_TT,DEST_STATE_TX,DEST_STATE_UT,DEST_STATE_VA,DEST_STATE_VI,DEST_STATE_VT,DEST_STATE_WA,DEST_STATE_WI,DEST_STATE_WV,DEST_STATE_WY
0,1,3,0,2475.0,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,3,0,2475.0,22,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,3,1,708.0,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,3,0,2176.0,18,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,3,1,1440.0,8,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Save preprocessed data

In [9]:
df_model.to_parquet('combined_preprocessed.parquet', index=False, compression='zstd')